In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from arch import arch_model
from scipy.stats import norm
from scipy.integrate import quad
import optuna
import warnings

warnings.filterwarnings("ignore")

# =====================================================================
# 1. VOLATILITY ESTIMATORS: GARCH(1,1) & LSTM
# =====================================================================

def fit_predict_garch11(returns: pd.Series) -> float:
    """
    Fits a standard GARCH(1,1) with Normal distribution and returns
    the next-day annualized volatility forecast.
    """
    # Scale returns by 100 for numerical stability in arch package
    scaled_returns = returns * 100.0
    model = arch_model(scaled_returns, vol="Garch", p=1, q=1, dist="normal", rescale=False)
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=1)
    next_day_var_scaled = forecast.variance.iloc[-1, 0]
    next_day_vol_daily = np.sqrt(next_day_var_scaled) / 100.0
    return float(next_day_vol_daily * np.sqrt(252))


class PyTorchLSTMVol(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return torch.relu(out)  # Ensure non-negative target volatility


def _create_sequences(features: np.ndarray, targets: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(len(features) - lookback):
        X.append(features[i : i + lookback])
        y.append(targets[i + lookback])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def optimize_and_train_lstm(
    feature_df: pd.DataFrame,
    target_series: pd.Series,
    n_trials: int = 15,
    seed: int = 42
) -> tuple[nn.Module, dict, float]:
    """
    Optimizes PyTorch LSTM hyperparameters with Optuna and trains the final model.
    Returns: (trained_model, best_params, next_day_forecast_vol)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    features = feature_df.values
    targets = target_series.values

    # 80/20 train-validation chronological split
    train_size = int(len(features) * 0.8)

    def objective(trial: optuna.Trial):
        lookback = trial.suggest_int("lookback", 10, 40, step=5)
        hidden_dim = trial.suggest_int("hidden_dim", 16, 64, step=8)
        num_layers = trial.suggest_int("num_layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.1, 0.4, step=0.1)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

        X, y = _create_sequences(features, targets, lookback)
        X_train, y_train = X[:train_size], y[:train_size]
        X_val, y_val = X[train_size:], y[train_size:]

        train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1))
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)

        model = PyTorchLSTMVol(X.shape[2], hidden_dim, num_layers, dropout)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.MSELoss()

        model.train()
        for epoch in range(15):  # Fast budget per trial
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                pred = model(batch_x)
                loss = criterion(pred, batch_y)
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(torch.tensor(X_val)).squeeze().numpy()
            # Quasi-Likelihood (QLIKE) loss: log(pred) + y / pred
            eps = 1e-6
            val_preds = np.clip(val_preds, eps, None)
            qlike = np.mean(np.log(val_preds) + y_val / val_preds)

        return qlike

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    best_p = study.best_params
    lookback = best_p["lookback"]

    # Re-train on full sequence history with optimal hyperparameters
    X_full, y_full = _create_sequences(features, targets, lookback)
    full_ds = TensorDataset(torch.tensor(X_full), torch.tensor(y_full).unsqueeze(1))
    full_loader = DataLoader(full_ds, batch_size=best_p["batch_size"], shuffle=False)

    best_model = PyTorchLSTMVol(
        X_full.shape[2], best_p["hidden_dim"], best_p["num_layers"], best_p["dropout"]
    )
    optimizer = torch.optim.Adam(best_model.parameters(), lr=best_p["lr"])
    criterion = nn.MSELoss()

    best_model.train()
    for epoch in range(25):
        for bx, by in full_loader:
            optimizer.zero_grad()
            pred = best_model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()

    best_model.eval()
    with torch.no_grad():
        last_window = torch.tensor(features[-lookback:], dtype=torch.float32).unsqueeze(0)
        next_day_forecast = float(best_model(last_window).item())

    return best_model, best_p, next_day_forecast


# =====================================================================
# 2. PRICING MODELS: BLACK-76 & SCHWARTZ-SMITH JUMP DIFFUSION
# =====================================================================

def black_76_price(
    F: float, K: float, T: float, r: float, sigma: float, option_type: str = "call"
) -> float:
    """
    Standard Black-76 formula for pricing European commodity futures options.
    """
    if T <= 0 or sigma <= 0:
        return max(0.0, (F - K) if option_type.lower() == "call" else (K - F))

    d1 = (np.log(F / K) + 0.5 * (sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    disc = np.exp(-r * T)

    if option_type.lower() == "call":
        return float(disc * (F * norm.cdf(d1) - K * norm.cdf(d2)))
    elif option_type.lower() == "put":
        return float(disc * (K * norm.cdf(-d2) - F * norm.cdf(-d1)))
    else:
        raise ValueError("option_type must be 'call' or 'put'")


def sstfmwj_price(
    F: float,
    K: float,
    T: float,
    r: float,
    sigma_base: float,
    kappa: float = 0.85,
    sigma_chi: float = 0.28,
    lambda_jump: float = 0.40,
    mu_jump: float = -0.05,
    sigma_jump: float = 0.15,
    option_type: str = "call",
    max_jumps: int = 20,
) -> float:
    """
    Schwartz-Smith Two-Factor Model With Jumps (SSTFMWJ).
    Uses Merton-style Poisson jump conditioning over the two-factor state variance.
    """
    if T <= 0:
        return max(0.0, (F - K) if option_type.lower() == "call" else (K - F))

    # Variance contribution of short-term state deviation (chi)
    var_short_term = (sigma_chi**2 / (2 * kappa)) * (1 - np.exp(-2 * kappa * T))
    # Combined effective diffusive variance over maturity
    integrated_diffusive_var = (sigma_base**2) * T + var_short_term

    # Jump compensator to maintain martingale condition for forward price
    k_drift = np.exp(mu_jump + 0.5 * (sigma_jump**2)) - 1.0
    lambda_prime = lambda_jump * (1.0 + k_drift)

    total_price = 0.0

    for n in range(max_jumps):
        prob_n = (np.exp(-lambda_prime * T) * (lambda_prime * T) ** n) / np.math.factorial(n)
        adjusted_variance = integrated_diffusive_var + n * (sigma_jump**2)
        sigma_n = np.sqrt(max(adjusted_variance / T, 1e-8))

        # Expected forward adjusted for n jumps
        F_n = F * np.exp(n * mu_jump + 0.5 * n * (sigma_jump**2) - n * np.log(1.0 + k_drift))

        p_n = black_76_price(F=F_n, K=K, T=T, r=r, sigma=sigma_n, option_type=option_type)
        total_price += prob_n * p_n

    return float(total_price)


# =====================================================================
# 3. END-TO-END WORKFLOW PIPELINE
# =====================================================================

def run_pricing_pipeline(
    underlying_df: pd.DataFrame,
    options_chain_df: pd.DataFrame,
    target_date: str,
    risk_free_rate: float = 0.045,
    optuna_trials: int = 15,
) -> pd.DataFrame:
    """
    Runs full parameter estimation, Optuna-driven LSTM calibration,
    GARCH forecasting, and option contract pricing.

    underlying_df requirements:
      - DatetimeIndex or 'date' column
      - 'close': Underlying asset close price
      - Additional feature columns (e.g. macro indicators, OVX, returns)
    options_chain_df requirements:
      - 'date': Quote evaluation date
      - 'strike': Strike price K
      - 'forward': Forward price F
      - 'dte': Days to expiration (used to calculate T = dte / 365)
      - 'type': 'call' or 'put'
    """
    df = underlying_df.copy()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    df = df.sort_index().loc[:target_date]

    # Calculate log returns and 20-day annualized realized target volatility
    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    df["target_vol"] = df["log_ret"].rolling(20).std() * np.sqrt(252)
    df = df.dropna()

    # 1. GARCH(1,1) Forecast
    vol_garch = fit_predict_garch11(df["log_ret"])

    # 2. Optuna-calibrated LSTM Forecast
    feature_cols = [c for c in df.columns if c not in ["target_vol"]]
    _, best_params, vol_lstm = optimize_and_train_lstm(
        feature_df=df[feature_cols],
        target_series=df["target_vol"],
        n_trials=optuna_trials,
    )

    print(f"[{target_date}] Volatility Forecasts:")
    print(f"  GARCH(1,1) Annualized Vol: {vol_garch:.4f}")
    print(f"  Optuna LSTM Annualized Vol: {vol_lstm:.4f}")

    # 3. Filter options for the valuation date
    chain = options_chain_df[options_chain_df["date"] == target_date].copy()
    if chain.empty:
        raise ValueError(f"No options found for evaluation date: {target_date}")

    results = []
    for _, row in chain.iterrows():
        F = float(row["forward"])
        K = float(row["strike"])
        T = float(row["dte"]) / 365.0
        opt_type = str(row["type"]).lower()

        # Pricing with GARCH Vol
        b76_garch = black_76_price(F, K, T, risk_free_rate, vol_garch, opt_type)
        sstf_garch = sstfmwj_price(F, K, T, risk_free_rate, vol_garch, option_type=opt_type)

        # Pricing with LSTM Vol
        b76_lstm = black_76_price(F, K, T, risk_free_rate, vol_lstm, opt_type)
        sstf_lstm = sstfmwj_price(F, K, T, risk_free_rate, vol_lstm, option_type=opt_type)

        results.append({
            "date": target_date,
            "strike": K,
            "forward": F,
            "dte": row["dte"],
            "type": opt_type,
            "b76_garch": b76_garch,
            "sstfmwj_garch": sstf_garch,
            "b76_lstm": b76_lstm,
            "sstfmwj_lstm": sstf_lstm,
        })

    return pd.DataFrame(results)


# =====================================================================
# 4. REPRODUCIBLE DEMO RUN
# =====================================================================

if __name__ == "__main__":
    # Generate synthetic time-series with macro variables
    dates = pd.date_range("2024-01-01", "2026-10-31", freq="B")
    np.random.seed(42)

    synthetic_returns = np.random.normal(0.0002, 0.02, size=len(dates))
    synthetic_prices = 70.0 * np.exp(np.cumsum(synthetic_returns))

    underlying_data = pd.DataFrame(
        {
            "date": dates,
            "close": synthetic_prices,
            "ovx_proxy": np.random.uniform(25, 45, size=len(dates)),
            "dxy_return": np.random.normal(0, 0.005, size=len(dates)),
        }
    )

    # Synthetic options chain on target date
    eval_date = "2026-10-30"
    forward_ref = float(underlying_data.loc[underlying_data["date"] == eval_date, "close"].iloc[0])

    options_data = pd.DataFrame(
        {
            "date": [eval_date] * 4,
            "forward": [forward_ref] * 4,
            "strike": [forward_ref - 5, forward_ref, forward_ref + 5, forward_ref],
            "dte": [30, 30, 30, 60],
            "type": ["call", "call", "call", "put"],
        }
    )

    priced_chain = run_pricing_pipeline(
        underlying_df=underlying_data,
        options_chain_df=options_data,
        target_date=eval_date,
        risk_free_rate=0.045,
        optuna_trials=5,
    )

    print("\nOption Pricing Matrix:")
    print(priced_chain.to_string(index=False))